# Retrain Sign-O-Text: 17 Classes

Retrains the LSTM model on 17 sign classes:
**hello, thanks, Father, Mother, Yes, No, Help, A, B, C, D, E, F, G, H, I, J**

## Setup
1. Run in **Google Colab with T4 GPU** (Runtime > Change runtime type > T4 GPU)
2. Upload your .npy data files to Google Drive
3. Mount Drive below.


In [ ]:
!pip install -q tensorflow==2.17.0 numpy scikit-learn matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
print('TensorFlow version:', tf.__version__)


In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/sign_o_text_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data directory:', DATA_DIR)
dirs = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
print('Found class directories:', dirs)


In [ ]:
SEQUENCE_LENGTH = 30
FEATURE_LENGTH = 126

CLASSES = ['hello', 'thanks', 'Father', 'Mother', 'Yes', 'No', 'Help',
           'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']

def load_data(data_dir):
    X, y = [], []
    for idx, class_name in enumerate(CLASSES):
        class_dir = data_dir / class_name
        if not class_dir.exists():
            print(f'MISSING: {class_name}')
            continue
        npy_files = list(class_dir.glob('*.npy')) + list(class_dir.glob('*.npz'))
        valid = 0
        for f in npy_files:
            data = np.load(f)
            if f.suffix == '.npz':
                data = data[list(data.keys())[0]]
            if data.shape == (SEQUENCE_LENGTH, FEATURE_LENGTH):
                X.append(data)
                y.append(idx)
                valid += 1
        print(f'{class_name}: {len(npy_files)} files, {valid} valid')
    X = np.array(X)
    y = np.array(y)
    print(f'Total: {len(X)} sequences, {len(np.unique(y))}/{len(CLASSES)} classes')
    return X, y

X, y = load_data(DATA_DIR)
print('Input shape:', X.shape)
print(f'Memory: {X.nbytes / 1024 / 1024:.1f} MB (float16 saves 2x vs float32)')


In [ ]:
def augment_sequence(seq, num_augments=50):
    results = [seq]
    for _ in range(num_augments - 1):
        aug = seq.copy()
        aug += np.random.normal(0, 0.02, seq.shape)
        scale = 1.0 + np.random.uniform(-0.05, 0.05)
        aug *= scale
        shift = np.random.uniform(-0.03, 0.03, (1, seq.shape[1]))
        aug += shift
        results.append(aug)
    return np.array(results)

def augment_dataset(X, y, n=50):
    X_aug, y_aug = [], []
    for i in range(len(X)):
        augs = augment_sequence(X[i], n)
        X_aug.extend(augs)
        y_aug.extend([y[i]] * len(augs))
    return np.array(X_aug), np.array(y_aug)

print(f'Augmenting 50x (from {len(X)} to {len(X)*50} sequences)...')
X_aug, y_aug = augment_dataset(X, y, n=50)
print(f'After augmentation: {X_aug.shape}')


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug, test_size=0.15, random_state=42, stratify=y_aug
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

NUM_CLASSES = len(CLASSES)

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, FEATURE_LENGTH)),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/best_action_17.keras',
    monitor='val_accuracy', save_best_only=True, mode='max'
)
early_stop = EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=150, batch_size=32,
    callbacks=[checkpoint, early_stop, lr_scheduler],
    verbose=1
)


In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc:.3f} ({acc*100:.1f}%)')
print('Per-class accuracy:')
cm = confusion_matrix(y_test, y_pred)
for i, name in enumerate(CLASSES):
    if i < cm.shape[0] and cm[i].sum() > 0:
        ca = cm[i,i] / cm[i].sum()
        print(f'{name:8s}: {ca:.2f} ({cm[i,i]}/{cm[i].sum()})')


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training Accuracy')
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss')
plt.tight_layout()
plt.show()


In [ ]:
final_path = '/content/drive/MyDrive/action_17_classes.keras'
model.save(final_path)
print(f'Model saved to: {final_path}')
import json
with open('/content/drive/MyDrive/classes_17.json', 'w') as f:
    json.dump(CLASSES, f, indent=2)
print('Class order saved.')


In [ ]:
from google.colab import files
files.download(final_path)
print('Done! Copy action_17_classes.keras to your project model/ folder')


## Summary
Data: WLASL (hello/thanks/Father/Mother/Yes/No/Help) + Google ASL (A-J)
Augmentation: 50x expansion via noise, scaling, shift
Output: Download action_17_classes.keras to your project
